# Structured Output with LangChain and Pydantic

This notebook demonstrates how to enforce structured JSON output from LLMs using Pydantic schemas with LangChain.

## Learning Objectives
- Define data schemas using Pydantic models
- Enforce structured output from LLMs
- Extract complex nested data structures
- Validate and type-check LLM responses

## Related Theoretical Content
- [AI Fundamentals](../../notes/03-ai/README.md)
- [Prompt Engineering](../../notes/03-ai/02-prompt_engineering/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import API Keys and Dependencies

Import the centralized API key management and required libraries.

In [ ]:
import os
from api_keys import get_api_key

# Set environment variable for OpenAI
os.environ["OPENAI_API_KEY"] = get_api_key('openai')

print(" API keys loaded")

## 1. Define Pydantic Schema

Pydantic models define the structure of data we expect from the LLM. Each field has:
- **Type annotation**: Specifies the data type (str, int, list, etc.)
- **Field description**: Helps the LLM understand what information to extract

This ensures type safety and automatic validation.

In [ ]:
from pydantic import BaseModel, Field

# Define the expected output schema
class PatientRecord(BaseModel):
    Name: str = Field(description="The patient's full name")
    Age: int = Field(description="The patient's age in years")
    Diagnosis: str = Field(description="The patient's medical diagnosis")
    Medication: list[str] = Field(description="List of medications the patient is taking")

print("Schema defined:")
print(PatientRecord.model_json_schema())

## 2. Initialize LangChain with Structured Output

LangChain's `with_structured_output()` method instructs the LLM to format its response according to our Pydantic schema.

**Key Concepts:**
- **ChatOpenAI**: LangChain wrapper for OpenAI models
- **Temperature=0**: Makes output deterministic for extraction tasks
- **with_structured_output()**: Enforces schema compliance

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the LLM with low temperature for consistency
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Bind the schema to the LLM
llm_with_structured_output = llm.with_structured_output(PatientRecord)

print(" LLM configured with structured output")

## 3. Extract Structured Data

Now we can send unstructured text and receive a validated Pydantic object. The LLM automatically:
1. Identifies relevant information
2. Maps it to schema fields
3. Formats it as valid JSON
4. Returns a type-safe Python object

In [ ]:
# Input: unstructured patient information
patient_details = "Patient name is Jane Smith, she is 45 years old and has been diagnosed with hypertension. She is currently taking Lisinopril."

# Extract structured data
response = llm_with_structured_output.invoke(patient_details)

# Response is now a validated Pydantic object
print("Extracted Patient Record:")
print(f"Name: {response.Name}")
print(f"Age: {response.Age}")
print(f"Diagnosis: {response.Diagnosis}")
print(f"Medications: {', '.join(response.Medication)}")

print("\nAs JSON:")
print(response.model_dump_json(indent=2))

## 4. Complex Nested Schema Example

Pydantic supports nested models for complex data structures. This is useful for extracting hierarchical information.

In [ ]:
# Define nested schemas
class Address(BaseModel):
    street: str = Field(description="Street address")
    city: str = Field(description="City name")
    state: str = Field(description="State or province")
    zip_code: str = Field(description="Postal code")

class EmergencyContact(BaseModel):
    name: str = Field(description="Contact person's name")
    relationship: str = Field(description="Relationship to patient")
    phone: str = Field(description="Phone number")

class DetailedPatientRecord(BaseModel):
    patient_name: str = Field(description="Patient's full name")
    age: int = Field(description="Patient's age")
    address: Address = Field(description="Patient's address")
    emergency_contact: EmergencyContact = Field(description="Emergency contact information")
    allergies: list[str] = Field(description="List of known allergies")

# Configure LLM with nested schema
llm_detailed = llm.with_structured_output(DetailedPatientRecord)

# Extract from complex text
detailed_info = """
Patient John Williams, age 52, lives at 123 Oak Street in Springfield, Illinois, 62701.
His emergency contact is his wife Mary Williams who can be reached at 555-0123.
He is allergic to penicillin and shellfish.
"""

detailed_response = llm_detailed.invoke(detailed_info)

print("Detailed Patient Record (Nested Structure):")
print(detailed_response.model_dump_json(indent=2))

## 5. Error Handling and Validation

Pydantic automatically validates data types. If the LLM returns invalid data, we get clear validation errors.

In [ ]:
from pydantic import ValidationError

# Try extracting from incomplete information
incomplete_info = "There's a patient named Bob."

try:
    result = llm_with_structured_output.invoke(incomplete_info)
    print("Extracted:")
    print(result.model_dump_json(indent=2))
except ValidationError as e:
    print("Validation Error:")
    print(e)
except Exception as e:
    print(f"Extraction handled missing data: {e}")

## 6. Multiple Entities Extraction

Extract multiple records from text containing several entities.

In [ ]:
class PatientList(BaseModel):
    patients: list[PatientRecord] = Field(description="List of patient records")

llm_multi = llm.with_structured_output(PatientList)

multi_patient_text = """
Today we saw three patients:
1. Jane Smith, 45 years old, diagnosed with hypertension, taking Lisinopril
2. Bob Johnson, 62 years old, diagnosed with diabetes, taking Metformin and Insulin
3. Alice Chen, 38 years old, diagnosed with asthma, taking Albuterol
"""

multi_response = llm_multi.invoke(multi_patient_text)

print(f"Extracted {len(multi_response.patients)} patients:")
for i, patient in enumerate(multi_response.patients, 1):
    print(f"\nPatient {i}:")
    print(f"  Name: {patient.Name}")
    print(f"  Age: {patient.Age}")
    print(f"  Diagnosis: {patient.Diagnosis}")
    print(f"  Medications: {', '.join(patient.Medication)}")

## Key Takeaways

1. **Type Safety**: Pydantic ensures extracted data matches expected types
2. **Automatic Validation**: Invalid data is caught immediately
3. **Nested Structures**: Complex hierarchical data can be extracted
4. **Developer Experience**: IDE autocomplete and type hints work with Pydantic models
5. **Reliability**: Structured output is more reliable than parsing free-form text

## Next Steps

- [03-rag-basics.ipynb](03-rag-basics.ipynb): Build retrieval systems for context-aware responses
- [04-advanced-rag.ipynb](04-advanced-rag.ipynb): Advanced RAG with document splitting and vector stores